In [ ]:
import shap
import numpy as np
import joblib
import pandas as pd
import matplotlib.pyplot as plt

RESULTS_DIR = r'C:\Users\GHANSHYAM\Desktop\voice-clone-detector\results'
SAVE_DIR    = r'C:\Users\GHANSHYAM\Desktop\voice-clone-detector\data\processed'

# Models load
xgb_model = joblib.load(f'{RESULTS_DIR}\\xgboost_model.pkl')
scaler    = joblib.load(f'{RESULTS_DIR}\\scaler.pkl')

# Data load
X_dev = np.load(f'{SAVE_DIR}\\X_dev.npy')
y_dev = np.load(f'{SAVE_DIR}\\y_dev.npy')
X_dev_sc = scaler.transform(X_dev)

# Feature names
feature_names = (
    [f'MFCC_{i}'    for i in range(40)] +
    ['Rolloff', 'ZCR'] +
    [f'Chroma_{i}'  for i in range(12)] +
    [f'Contrast_{i}'for i in range(7)]
)

print(f"Features: {len(feature_names)}")
print("Ready for SHAP ✅")

In [ ]:
# Sample 1000 rows — SHAP pe full dataset slow hoga
sample_idx = np.random.choice(len(X_dev_sc), 1000, replace=False)
X_sample   = X_dev_sc[sample_idx]
y_sample   = y_dev[sample_idx]

# TreeExplainer — XGBoost ke liye fast
explainer   = shap.TreeExplainer(xgb_model)
shap_values = explainer.shap_values(X_sample)

print(f"SHAP values shape: {shap_values.shape}")

# Global Summary Plot
plt.figure(figsize=(10, 8))
shap.summary_plot(
    shap_values, X_sample,
    feature_names=feature_names,
    show=False
)
plt.title("SHAP — Global Feature Importance", fontsize=13)
plt.tight_layout()
plt.savefig(r'C:\Users\GHANSHYAM\Desktop\voice-clone-detector\results\figures\shap_global.png',
            dpi=150, bbox_inches='tight')
plt.show()
print("SHAP global plot saved ✅")

In [ ]:
# Top 10 features — bar plot
shap_mean = np.abs(shap_values).mean(axis=0)
top10_idx  = np.argsort(shap_mean)[-10:][::-1]
top10_names = [feature_names[i] for i in top10_idx]
top10_vals  = shap_mean[top10_idx]

plt.figure(figsize=(10, 6))
bars = plt.barh(top10_names[::-1], top10_vals[::-1], color='steelblue', alpha=0.85)
plt.xlabel('Mean |SHAP value|')
plt.title('Top 10 Most Important Features — Voice Clone Detection', fontsize=13)

for bar, val in zip(bars, top10_vals[::-1]):
    plt.text(val + 0.05, bar.get_y() + bar.get_height()/2,
             f'{val:.3f}', va='center', fontsize=9)

plt.tight_layout()
plt.savefig(r'C:\Users\GHANSHYAM\Desktop\voice-clone-detector\results\figures\shap_top10.png',
            dpi=150, bbox_inches='tight')
plt.show()

print("\nTop 10 Features:")
for i, (name, val) in enumerate(zip(top10_names, top10_vals)):
    print(f"{i+1:2}. {name:15} — {val:.4f}")

In [ ]:
# Local explanation — ek spoof file pe
spoof_idx  = np.where(y_sample == 1)[0][0]
spoof_feat = X_sample[spoof_idx:spoof_idx+1]

plt.figure(figsize=(10, 5))
shap.waterfall_plot(
    shap.Explanation(
        values        = shap_values[spoof_idx],
        base_values   = explainer.expected_value,
        data          = spoof_feat[0],
        feature_names = feature_names
    ),
    show=False
)
plt.title("Local Explanation — Single Spoof Sample", fontsize=12)
plt.tight_layout()
plt.savefig(r'C:\Users\GHANSHYAM\Desktop\voice-clone-detector\results\figures\shap_local.png',
            dpi=150, bbox_inches='tight')
plt.show()
print("Local explanation saved ✅")